<h5> Imports de librerías de pyspark </h5>

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.dataframe import DataFrame
from pyspark.errors import AnalysisException
from delta.tables import *
from typing import Dict, List

In [0]:
input_catalog = "bronze"
output_catalog = "silver"
schema = "sales"
table_name = f"{output_catalog}.{schema}.total_sales"
force_history = False

<h5> Configuración de la aplicación de Spark </h5>

In [0]:
spark = (
    SparkSession.builder
    .appName('SalesPipelineSilverApp')
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

<h5> Extracción de datos al volume </h5>

In [0]:
root_path = '/Volumes/bronze/origin_system/raw_data'

df_productos = spark.read.format('csv').options(**{'header' : 'true', 'inferSchema': 'true'}).load(f'{root_path}/Productos.csv') 
df_ventas = spark.read.format('csv').options(**{'header' : 'true', 'inferSchema': 'true'}).load(f'{root_path}/Venta.csv') 
df_clientes = spark.read.format('csv').options(**{'header': 'true', 'inferSchema': 'true', 'sep': ';'}).load(f'{root_path}/Clientes.csv')
df_sucursales = spark.read.format('csv').options(**{'header' : 'true', 'inferSchema': 'true', 'sep': ';'}).load(f'{root_path}/Sucursales.csv')

<h5> Verificar schemas de dataframe origen </h5>

In [0]:
df_productos.printSchema() 
df_ventas.printSchema()     
df_clientes.printSchema()  
df_sucursales.printSchema() 

<h5> Comprobamos clave natural de dataframes </h5>
<p> Para considerar una columna clave tiene que ser </p>
<ul>
  <li> Única  </li>
  <li> No Nula </li>
</ul>

In [0]:
def validate_natural_key(dataframe_and_key: Dict[DataFrame, List[str]]) -> None:
    import logging
    from pyspark.sql import DataFrame
    from pyspark.sql.utils import AnalysisException
    from pyspark.sql.functions import col, concat_ws
    
    logger = logging.getLogger(__name__)
    logging.basicConfig(level=logging.INFO)
    
    validated_keys = []
    
    for df, key_columns in dataframe_and_key.items():
        try:
            null_condition = ' OR '.join([f"{col_name} IS NULL" for col_name in key_columns])
            null_count = df.filter(null_condition).count()
            duplicated_count = df.groupBy(*key_columns).count().filter("count > 1").count()
            
            if null_count > 0 or duplicated_count > 0:
                raise ValueError(
                    f"Hay {null_count} registros nulos y {duplicated_count} duplicados "
                    f"para la clave {key_columns}."
                )
            else:
                logger.info(f"La clave {key_columns} es válida.")
                validated_keys.append(key_columns)
                
        except AnalysisException as err:
            logger.error(f"Error al validar clave {key_columns}: {err}")
            raise err
    
    logger.info(f"Todas las claves validadas: {validated_keys}")



In [0]:
validate_natural_key({
    df_productos : ['ID_PRODUCTO'],
    df_clientes  : ['ID'], 
    df_sucursales:  ['ID'],
    df_ventas : ['IdVenta']
})

In [0]:
df_productos_normalized = (
    df_productos.withColumnRenamed('Precio ','precio_producto')
    .drop('Precio ')
)
df_clientes_normalized = (
    df_clientes.withColumnRenamed('ID', 'client_id')
    .withColumnRenamed('Provincia','provicia_cliente')
    .withColumnRenamed('Localidad', 'localidad_cliente')
    .drop('ID', 'Provincia', 'Localidad')
)
df_sucursales_normalized = (
    df_sucursales.withColumnRenamed('ID', 'sucursal_id')
    .withColumnRenamed('Provincia','provicia_sucursal')
    .withColumnRenamed('Localidad', 'localidad_sucursal')
    .drop('ID','Provincia', 'Localidad')
)



<h4> Joins para obtener las ventas totales </h4>

In [0]:
ventas_productos_df = (
    df_ventas.join(
        df_productos_normalized, on = df_ventas['IdProducto'] == df_productos_normalized['ID_PRODUCTO'], how = 'left')
)

ventas_productos_clientes = (
    ventas_productos_df.join(
        df_clientes_normalized, on = ventas_productos_df['IdCliente'] == df_clientes_normalized['client_id'], how = 'left')
)

ventas_productos_clientes_sucursales = (
    ventas_productos_clientes.join(
        df_sucursales_normalized, on = ventas_productos_clientes['IdSucursal'] == df_sucursales_normalized['sucursal_id'], how = 'left'
    )
)


In [0]:
ventas_productos_clientes_sucursales.display()

##### Preparamos dataframe para generar tabla en silver
<p>Convertimos todas las columnas a mayúsculas</p> 
<p>Creamos las fechas de sistema</p>

In [0]:
def prepare_silver_dataframe(silver_df: DataFrame) -> DataFrame:
    from pyspark.sql.functions import col, current_timestamp, from_utc_timestamp, lit

    processed_df = silver_df
    for column in silver_df.columns:
        processed_df = processed_df.withColumnRenamed(column, column.upper())

    return (
        processed_df.withColumn("CREATED_AT", from_utc_timestamp(current_timestamp(), "America/Argentina/Buenos_Aires"))
        .withColumn("UPDATED_AT", from_utc_timestamp(current_timestamp(), "America/Argentina/Buenos_Aires"))
        .withColumn("DELETED", lit(False).cast('boolean'))
        .withColumn("DELETED_AT", lit(None).cast('timestamp'))
    )

processed_df = prepare_silver_dataframe(...)
